<a href="https://colab.research.google.com/github/tinemyumi/saude-mental-datasus/blob/main/notebooks/1.download_sihsus_ibge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Download SIH SUS e IBGE**

**Conteúdo**
- Download dos dados do datasus, concatenação dos arquivos .parquet em meses para um arquivo ano.parquet e transferência permamente desses dados para o Google Drive;
- Download dos arquivos de população do IBGE

**Autor:** Larissa Tinem


# **Conexão com o Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Baixando dados do Datasus (SIH-SUS) via  pysus**

Baixando dados do Datasus(SIH-SUS, Estado de São Paulo, do ano de 2015 a 2025) através da biblioteca pysus.

In [ ]:
# Instalação da biblioteca pysus
!pip install pysus

In [ ]:
# Código utilizado para realizar o download da base da de dados do datasus, a respeito do SIH-SUS

from pysus import SIH
import os
import pandas as pd

# 1. Carregando a biblioteca PySUS, os arquivos do datasus
sih = SIH().load()

# 2. Definição dos anos para realizar download (como exemplo o ano de 2015, mas pode ser uma lista dos anos desejáveis)
anos_para_baixar = [2025] # acrescentar os anos que queremos realizar o download
diretorio_base = '/content/dados_sih/'

# 3. Loop para processar cada ano
for ano in anos_para_baixar:
    diretorio_ano = os.path.join(diretorio_base, str(ano))
    os.makedirs(diretorio_ano, exist_ok=True)

    print(f"Buscando e baixando dados para o ano: {ano}...")

    # Buscando os arquivos para o ano atual e para todos os meses
    arquivos_ano = sih.get_files("RD", uf="sp", year=ano, month=list(range(1, 13)))

    # Baixando os arquivos para a pasta específica do ano
    sih.download(arquivos_ano, local_dir=diretorio_ano)

    print(f"Download para {ano} concluído na pasta: {diretorio_ano}")

print("\nProcesso de download para todos os anos concluído!")

In [ ]:
# Código de importação dados do content para o Google Drive

from google.colab import drive
import shutil

# Monta o seu Google Drive no Colab
drive.mount('/content/drive')

# Caminho da pasta de origem no Colab
caminho_origem = '/content/dados_sih/'

# Caminho de destino Google Drive
# A pasta principal Drive é '/content/drive/MyDrive/'
caminho_destino = '/content/drive/MyDrive/DadoDataSUS/dados_sih/'

print("Iniciando a transferência dos arquivos para o Google Drive...")

try:
    shutil.move(caminho_origem, caminho_destino)
    print("\nTransferência concluída com sucesso!")
    print(f"Os arquivos agora estão salvos em: {caminho_destino}")

except FileNotFoundError:
    print("\nErro: A pasta de origem não foi encontrada. Verifique se o caminho está correto.")

except Exception as e:
    print(f"\nOcorreu um erro durante a transferência: {e}")


# **Unir meses do ano em único .parquet**

Este código realiza a leitura de todos os arquivos .parquet mensais de um ano específico, concatena os dados em um único DataFrame e exporta o resultado consolidado em um único arquivo .parquet, facilitando a análise e o gerenciamento das informações

Data de realização: 08/10/2025 Autor: Larissa Tinem

In [ ]:
!pip install fastparquet
!pip install pyarrow

In [ ]:
import os
import pandas as pd

#Escolha do ano
ano = 2025

# Caminho base
caminho_base = "/content/drive/MyDrive/DadoDataSUS/dados_sih/dados_sih"
pasta_ano = os.path.join(caminho_base, str(ano))

# Caminho destino
caminho_destino = "/content/drive/MyDrive/DadoDataSUS/dados_sih/dados_datasus"
os.makedirs(caminho_destino, exist_ok=True)

# Lista todos os arquivos parquet do ano
arquivos = sorted([os.path.join(pasta_ano, f) for f in os.listdir(pasta_ano) if f.endswith(".parquet")])

print(f"🔄 Concatenando {len(arquivos)} arquivos do ano {ano}...")

# Concatena todos os meses
dfs = [pd.read_parquet(arq) for arq in arquivos]
df_anual = pd.concat(dfs, ignore_index=True)

# Salva o arquivo consolidado
destino = os.path.join(caminho_destino, f"{ano}.parquet")
df_anual.to_parquet(destino, index=False)

print(f"✅ Arquivo do ano {ano} salvo em {destino}")


In [ ]:
import os
import pandas as pd
import gc

# CONFIGURAÇÕES
ano = 2024
caminho_base = '/content/drive/MyDrive/DadosSIA'
pasta_ano = os.path.join(caminho_base, str(ano))
caminho_destino = '/content/drive/MyDrive/DadosSIA/Concatenado'
os.makedirs(caminho_destino, exist_ok=True)

# Lista de arquivos parquet
arquivos = sorted([os.path.join(pasta_ano, f) for f in os.listdir(pasta_ano) if f.endswith(".parquet")])
print(f"🔍 {len(arquivos)} arquivos encontrados para o ano {ano}\n")

# LEITURA SEGURA
dfs = []
arquivos_ok = []
arquivos_ruins = []

for arq in arquivos:
    try:
        df_temp = pd.read_parquet(arq, engine='fastparquet')  # força engine alternativa
        dfs.append(df_temp)
        arquivos_ok.append(os.path.basename(arq))
        print(f"✅ OK: {os.path.basename(arq)} ({len(df_temp)} linhas)")
    except Exception as e:
        arquivos_ruins.append((os.path.basename(arq), str(e)))
        print(f"❌ ERRO: {os.path.basename(arq)} → {type(e).__name__}")

    # limpeza de memória
    del df_temp
    gc.collect()

# CONCATENA E SALVA
if dfs:
    df_anual = pd.concat(dfs, ignore_index=True)
    destino = os.path.join(caminho_destino, f"{ano}.parquet")
    df_anual.to_parquet(destino, index=False)
    print(f"\n✅ Arquivo final salvo em: {destino}")
    print(f"📊 Total de registros: {len(df_anual)}")
else:
    print("\n⚠️ Nenhum arquivo válido foi carregado!")

# RELATÓRIO FINAL
print("\n--- RESUMO ---")
print(f"✅ Arquivos OK: {len(arquivos_ok)}")
print(f"❌ Arquivos com erro: {len(arquivos_ruins)}")

if arquivos_ruins:
    print("\nArquivos problemáticos:")
    for nome, erro in arquivos_ruins:
        print(f" - {nome}: {erro[:120]}...")


# **Baixando dados do IBGE**
O procedimento de extração dos dados foi realizado com base no guia disponível em: https://dkko.me/posts/coletando-ibge-sidra-populacao-municipios/

In [ ]:
!pip install sidrapy

In [ ]:
from pathlib import Path
import requests
import sidrapy


In [ ]:
# Função para obter os períodos disponíveis de uma tabela
def get_periodos(agregado: str):
  url = f'https://servicodados.ibge.gov.br/api/v3/agregados/{agregado}/periodos'
  response = requests.get(url)
  return response.json()

In [ ]:
# Função para baixar uma tabela do sidra
def download_table(
    sidra_tabela: str,
    territorial_level: str,
    ibge_territorial_code: str,
    variable: str = 'allxp',
    classifications: dict = None,
    data_dir: Path = Path('data'),
) -> list[Path]:
  """Download a SIDRA table in CSV format on temp_dir()

    Args:
        sidra_tabela (str): SIDRA table code
        territorial_level (str): territorial level code
        ibge_territorial_code (str): IBGE territorial code
        variable (str, optional): variable code. Defaults to None.
        classifications (dict, optional): classifications and categories codes.
            Defaults to None.

    Returns:
        list[Path]: list of downloaded files
    """
  filepaths = []
  periodos = get_periodos(sidra_tabela)
  for periodo in periodos:
        filename = f"{periodo['id']}.csv"
        dest_filepath = data_dir / filename
        dest_filepath.parent.mkdir(exist_ok=True, parents=True)
        if dest_filepath.exists():
            print("File already exists:", dest_filepath)
            continue
        print("Downloading", filename)
        df = sidrapy.get_table(
            table_code=sidra_tabela,  # Tabela SIDRA
            territorial_level=territorial_level,  # Nível de Municípios
            ibge_territorial_code=ibge_territorial_code,  # Territórios
            period=periodo["id"],  # Período
            variable=variable,  # Variáveis
            classifications=classifications,
        )
        df.to_csv(dest_filepath, index=False, encoding="utf-8")
        filepaths.append(dest_filepath)
  return filepaths

In [ ]:
# Diretório para salvar os arquivos
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Lista em python para colocar os caminhos dos arquivos baixados
files = []


In [ ]:
# Populacao Censos
sidra_tabela = "200"
territorial_level = "6"
ibge_territorial_code = "all"

files_census = download_table(
    sidra_tabela=sidra_tabela,
    territorial_level=territorial_level,
    ibge_territorial_code=ibge_territorial_code,
    variable="allxp",
    classifications={"2": "0", "1": "0", "58": "0"},
    data_dir=data_dir,
)
files.extend(files_census)


In [ ]:
# Populacao Censo 2022
sidra_tabela = "9514"
territorial_level = "6"
ibge_territorial_code = "all"

files_census_2022 = download_table(
    sidra_tabela=sidra_tabela,
    territorial_level=territorial_level,
    ibge_territorial_code=ibge_territorial_code,
    variable="allxp",
    classifications={"2": "6794", "287": "100362", "286": "113635"},
    data_dir=data_dir,
)
files.extend(files_census_2022)


In [ ]:
# Populacao Contagens
sidra_tabelas = (
    "305",
    "793",
)

for sidra_tabela in sidra_tabelas:
    files_counts = download_table(
        sidra_tabela=sidra_tabela,
        territorial_level=territorial_level,
        ibge_territorial_code=ibge_territorial_code,
        data_dir=data_dir,
    )
    files.extend(files_counts)


In [ ]:
# Populacao Estimativas
sidra_tabela = "6579"

files_estimates = download_table(
    sidra_tabela=sidra_tabela,
    territorial_level=territorial_level,
    ibge_territorial_code=ibge_territorial_code,
    data_dir=data_dir,
)
files.extend(files_estimates)


In [ ]:
# Consolidando arquivos
import pandas as pd

def read_file(filepath: Path, **read_csv_args) -> pd.DataFrame:
    print("Reading file", filepath)
    data = pd.read_csv(filepath, skiprows=1, na_values=["...", "-"], **read_csv_args)
    data = data.dropna(subset="Valor")
    return data

def refine(df: pd.DataFrame) -> pd.DataFrame:
    df = (
        df.dropna(subset="Valor")
        .rename(
            columns={
                "Ano": "ano",
                "Município (Código)": "id_municipio",
                "Valor": "populacao",
            }
        )
        .assign(pessoas=lambda x: x["populacao"].astype(int))
    )
    df[["nome_municipio", "sigla_uf"]] = df["Município"].str.split(" - ", expand=True)
    df = df.drop(columns="Município")
    df = df[["ano", "id_municipio", "nome_municipio", "sigla_uf", "populacao"]]
    return df


In [ ]:
df = refine(
    pd.concat(
        (
            read_file(file, usecols=("Ano", "Município (Código)", "Município", "Valor"))
            for file in files
        ),
        ignore_index=True,
    )
)


Reading file data/1970.csv
Reading file data/1980.csv
Reading file data/1991.csv
Reading file data/2000.csv
Reading file data/2010.csv
Reading file data/2022.csv
Reading file data/1996.csv
Reading file data/2007.csv
Reading file data/2001.csv
Reading file data/2002.csv
Reading file data/2003.csv
Reading file data/2004.csv
Reading file data/2005.csv
Reading file data/2006.csv
Reading file data/2008.csv
Reading file data/2009.csv
Reading file data/2011.csv
Reading file data/2012.csv
Reading file data/2013.csv
Reading file data/2014.csv
Reading file data/2015.csv
Reading file data/2016.csv
Reading file data/2017.csv
Reading file data/2018.csv
Reading file data/2019.csv
Reading file data/2020.csv
Reading file data/2021.csv
Reading file data/2024.csv
Reading file data/2025.csv


In [ ]:
print(df.head())


    ano  id_municipio   nome_municipio sigla_uf  populacao
0  1970       1100106    Guajará-Mirim       RO    27016.0
1  1970       1100205      Porto Velho       RO    84048.0
2  1970       1200104        Brasiléia       AC    12311.0
3  1970       1200203  Cruzeiro do Sul       AC    43584.0
4  1970       1200302            Feijó       AC    15768.0


In [ ]:
df.to_csv("/content/drive/MyDrive/Dados/populacao_municipios.csv", index=False, encoding="utf-8")
